# Assignment 4: Detección de Sarcasmo con Regresión Logística

En este notebook vamos a construir un clasificador capaz de detectar si un comentario de Reddit es sarcástico o no.

### ¿Por qué es difícil detectar el sarcasmo?
El sarcasmo es una forma de comunicación donde el significado **literal** y el significado **real** son opuestos. Por ejemplo:
- "¡Genial, otro lunes perfecto!" → probablemente sarcástico
- "Me encanta el café por las mañanas" → probablemente sincero

Para una máquina, distinguir estos casos es un reto enorme porque no tiene contexto cultural ni entonación.

### Dataset
Usamos el paper [A Large Self-Annotated Corpus for Sarcasm](https://arxiv.org/abs/1704.05579): más de 1 millón de comentarios de Reddit etiquetados automáticamente gracias al subreddit `/r/sarcasm` (cuando alguien publica ahí, está marcando explícitamente su comentario como sarcástico).

### Pipeline que vamos a construir
```
Texto crudo → TF-IDF (vectorización) → Regresión Logística → Predicción (sarcástico/no)
```

## 0. Importación de librerías

In [ ]:
# ── Librerías básicas de análisis de datos ──────────────────────────────────
import numpy as np          # operaciones numéricas y arrays
import pandas as pd         # manejo de datos tabulares (DataFrames)

# ── Visualización ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt   # gráficos estándar
import seaborn as sns             # gráficos estadísticos más bonitos

# ── NLP: Vectorización de texto ──────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
# TF-IDF convierte texto en números que reflejan la importancia de cada palabra

# ── Modelo de Machine Learning ────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
# Regresión Logística: a pesar del nombre, es un clasificador. Muy eficiente para texto.

# ── Pipeline: encadena pasos de preprocesado + modelo ────────────────────────
from sklearn.pipeline import Pipeline
# Pipeline asegura que el TF-IDF se ajuste SOLO con train, evitando data leakage

# ── Evaluación del modelo ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,          # % de predicciones correctas
    classification_report,   # precisión, recall y F1 por clase
    confusion_matrix          # matriz de confusión
)

# ── División train/test ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')  # silencia warnings molestos durante el entrenamiento

# Configuración general de gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ Librerías importadas correctamente")

## 1. Carga y exploración inicial del dataset

In [ ]:
# Cargamos el CSV con todos los comentarios de Reddit
# El dataset tiene ~1 millón de filas, así que puede tardar unos segundos
df = pd.read_csv("train-balanced-sarcasm.csv")

# Mostramos las primeras filas para entender qué columnas tenemos
df.head()

In [ ]:
# Información general del DataFrame:
# - Cuántas filas y columnas hay
# - Tipo de dato de cada columna
# - Cuántos valores NO nulos hay (si count < total → hay NaN)
df.info()

In [ ]:
# Descripción de las columnas más relevantes:
# ─────────────────────────────────────────────────────────────────
# label          → 1 = sarcástico, 0 = no sarcástico (nuestra VARIABLE OBJETIVO)
# comment        → texto del comentario de Reddit (nuestra VARIABLE PREDICTORA)
# author         → usuario de Reddit que escribió el comentario
# subreddit      → comunidad de Reddit donde se publicó
# score          → puntuación del comentario (votos positivos - negativos)
# parent_comment → comentario al que está respondiendo (contexto)
# ─────────────────────────────────────────────────────────────────

print(f"Total de comentarios: {len(df):,}")
print(f"Columnas: {df.columns.tolist()}")

## 2. Limpieza: Eliminar valores nulos (missings)

In [ ]:
# Primero contamos cuántos valores nulos hay en cada columna
# Los valores nulos (NaN) pueden causar errores en el modelo
print("Valores nulos por columna:")
print(df.isnull().sum())

# La columna 'comment' tiene algunos nulos: son filas donde no hay texto
# No podemos predecir el sarcasmo de un comentario vacío → los eliminamos

In [ ]:
# Eliminamos las filas con cualquier valor nulo
# inplace=True modifica el DataFrame directamente (sin crear uno nuevo)
n_antes = len(df)
df.dropna(inplace=True)
n_despues = len(df)

print(f"Filas antes de eliminar nulos: {n_antes:,}")
print(f"Filas después de eliminar nulos: {n_despues:,}")
print(f"Filas eliminadas: {n_antes - n_despues}")

# Verificamos que no quedan nulos
print("\n¿Quedan nulos?")
print(df.isnull().sum())

## 3. Comprobación del balance del dataset

Un dataset **balanceado** tiene aproximadamente el mismo número de ejemplos de cada clase. Esto es importante porque:
- Si hubiera 90% no-sarcástico y 10% sarcástico, un modelo "tonto" que siempre predice "no sarcástico" tendría 90% de accuracy... pero sería inútil.
- Con clases balanceadas, el accuracy es una métrica más honesta.

In [ ]:
# Contamos cuántos ejemplos hay de cada clase (0 y 1)
conteo = df['label'].value_counts()
print("Distribución de clases:")
print(conteo)
print(f"\nPorcentajes:")
print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
# Visualizamos el balance con un gráfico de barras
fig, ax = plt.subplots(figsize=(6, 4))

colores = ['#3498db', '#e74c3c']  # azul = no sarcástico, rojo = sarcástico
etiquetas = ['No sarcástico (0)', 'Sarcástico (1)']

bars = ax.bar(etiquetas, conteo.values, color=colores, edgecolor='black', width=0.5)

# Añadimos el número encima de cada barra para mayor claridad
for bar, valor in zip(bars, conteo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{valor:,}', ha='center', fontsize=12, fontweight='bold')

ax.set_title('Distribución de clases: ¿Está balanceado el dataset?', fontsize=14)
ax.set_ylabel('Número de comentarios')
ax.set_ylim(0, conteo.max() * 1.15)
plt.tight_layout()
plt.show()

print("\n✅ El dataset está perfectamente balanceado (50% / 50%)")

## 4. Análisis exploratorio del texto (EDA)

Antes de modelar, exploramos el texto para entender qué diferencia a los comentarios sarcásticos de los no sarcásticos.

In [ ]:
# ── Longitud de los comentarios ──────────────────────────────────────────────
# Calculamos el número de caracteres de cada comentario
# (una característica simple que puede ser informativa)
df['longitud'] = df['comment'].str.len()

# Estadísticas por clase
print("Longitud media de comentarios por clase:")
print(df.groupby('label')['longitud'].describe().round(1))
print("\nLabel 0 = No sarcástico | Label 1 = Sarcástico")

In [ ]:
# Visualizamos la distribución de longitudes por clase
# Usamos solo comentarios de hasta 500 caracteres para que el gráfico sea legible
fig, ax = plt.subplots(figsize=(11, 4))

for label, color, nombre in [(0, '#3498db', 'No sarcástico'), (1, '#e74c3c', 'Sarcástico')]:
    subset = df[df['label'] == label]['longitud']
    # Filtramos outliers extremos para mejor visualización
    subset = subset[subset <= 500]
    ax.hist(subset, bins=50, alpha=0.6, color=color, label=nombre, density=True)

ax.set_title('Distribución de longitud de comentarios por clase', fontsize=13)
ax.set_xlabel('Número de caracteres')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Ejemplos de comentarios sarcásticos vs no sarcásticos ────────────────────
# Ver ejemplos reales ayuda a entender el problema intuitivamente

pd.set_option('max_colwidth', 200)  # para ver el texto completo

print("=" * 70)
print("EJEMPLOS DE COMENTARIOS SARCÁSTICOS (label = 1):")
print("=" * 70)
for txt in df[df['label'] == 1]['comment'].sample(5, random_state=42).values:
    print(f"  ▶ {txt}")
    print()

print("=" * 70)
print("EJEMPLOS DE COMENTARIOS NO SARCÁSTICOS (label = 0):")
print("=" * 70)
for txt in df[df['label'] == 0]['comment'].sample(5, random_state=42).values:
    print(f"  ▶ {txt}")
    print()

In [ ]:
# ── Subreddits más frecuentes por clase ──────────────────────────────────────
# ¿Hay subreddits donde el sarcasmo es más común?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, titulo, color in [
    (axes[0], 0, 'Top 10 subreddits: No Sarcástico', '#3498db'),
    (axes[1], 1, 'Top 10 subreddits: Sarcástico',    '#e74c3c')
]:
    top = df[df['label'] == label]['subreddit'].value_counts().head(10)
    ax.barh(top.index[::-1], top.values[::-1], color=color, edgecolor='black')
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('Número de comentarios')

plt.tight_layout()
plt.show()

# Interpretación: si algunos subreddits aparecen mucho en sarcásticos,
# significa que ciertas comunidades son más propensas al sarcasmo

## 5. División en Train y Test

Dividimos el dataset en dos partes:
- **Train (80%)**: el modelo aprende con estos datos
- **Test (20%)**: evaluamos el modelo con datos que nunca ha visto

⚠️ Es crucial que el modelo NO vea los datos de test durante el entrenamiento. Si no, estaríamos haciendo "trampa" y el accuracy no sería real.

In [ ]:
# Separamos características (X) y etiquetas (y)
# X: el texto del comentario (lo que usará el modelo para predecir)
# y: la etiqueta (0 o 1) que queremos predecir
X = df['comment']
y = df['label']

# Dividimos en train (80%) y test (20%)
# random_state=42 → semilla aleatoria fija, para que siempre obtengamos la misma división
# stratify=y → asegura que train y test tengan el mismo % de cada clase
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para test
    random_state=42,     # reproducibilidad
    stratify=y           # mantiene el balance 50/50 en ambas particiones
)

print("Forma del conjunto de entrenamiento:")
print(f"  X_train: {X_train.shape}  →  {len(X_train):,} comentarios")
print(f"  y_train: {y_train.shape}")

print("\nForma del conjunto de test:")
print(f"  X_test:  {X_test.shape}   →  {len(X_test):,} comentarios")
print(f"  y_test:  {y_test.shape}")

# Verificamos que el balance se mantiene en ambas particiones
print("\n% de sarcásticos en train:", y_train.mean().round(3))
print("% de sarcásticos en test: ", y_test.mean().round(3))

## 6. Vectorización TF-IDF

Los modelos de ML no entienden texto directamente → hay que convertirlo en números.

### ¿Qué es TF-IDF?
**TF-IDF** = Term Frequency × Inverse Document Frequency

- **TF** (Term Frequency): ¿con qué frecuencia aparece la palabra en este comentario?
- **IDF** (Inverse Document Frequency): ¿qué tan rara es esta palabra en todo el corpus?

Palabras muy comunes en todos los textos ('the', 'is', 'a') tienen IDF bajo → poco informativas.  
Palabras raras pero presentes en un comentario concreto tienen IDF alto → muy informativas.

$$\text{TF-IDF}(t, d) = TF(t,d) \times \log\left(\frac{N}{df(t)}\right)$$

### Pipeline: ¿por qué usarlo?
Con `Pipeline`, el TF-IDF se ajusta (`.fit()`) **solo con train** y después transforma ambos. Así evitamos **data leakage** (que información del test "contamine" el entrenamiento).

In [ ]:
# ── Construcción del Pipeline ─────────────────────────────────────────────────
# Un Pipeline encadena pasos: primero vectoriza, luego clasifica.
# Cada paso tiene un nombre ('tfidf', 'clf') y un objeto de sklearn.

pipeline = Pipeline([
    # PASO 1: Vectorización TF-IDF
    ('tfidf', TfidfVectorizer(
        max_features=50000,   # usar las 50.000 palabras/bigramas más frecuentes
                              # (limitar evita matrices gigantes y ruido)
        ngram_range=(1, 2),   # unigramas ("good") Y bigramas ("not good")
                              # Los bigramas capturan negaciones importantes
        min_df=3,             # ignorar palabras que aparecen en <3 documentos
                              # (palabras rarísimas son ruido, no señal)
        sublinear_tf=True,    # aplica log(1 + tf) en vez de tf
                              # reduce el efecto de palabras muy repetidas
    )),

    # PASO 2: Clasificador — Regresión Logística
    ('clf', LogisticRegression(
        C=1.0,                # inverso de la regularización L2
                              # C alto = menos regularización (más flexible)
                              # C bajo = más regularización (más conservador)
        max_iter=1000,        # número máximo de iteraciones para converger
        solver='lbfgs',       # optimizador eficiente para datasets grandes
        n_jobs=-1,            # usa todos los núcleos del CPU disponibles
        random_state=42
    ))
])

print("Pipeline creado:")
print(pipeline)

## 7. Entrenamiento del modelo

In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────────
# pipeline.fit() hace dos cosas en secuencia:
# 1. tfidf.fit_transform(X_train) → aprende el vocabulario y vectoriza el texto
# 2. clf.fit(X_train_vectorizado) → entrena la Regresión Logística
# Solo se hace con X_train, NUNCA con X_test

print("Entrenando el modelo... (puede tardar unos minutos con 800k ejemplos)")

import time
inicio = time.time()

pipeline.fit(X_train, y_train)

fin = time.time()
print(f"✅ Entrenamiento completado en {fin - inicio:.1f} segundos")

## 8. Evaluación del modelo

In [ ]:
# ── Predicciones sobre el conjunto de TEST ────────────────────────────────────
# pipeline.predict() hace:
# 1. tfidf.transform(X_test) → vectoriza con el vocabulario ya aprendido en train
# 2. clf.predict() → clasifica cada comentario como 0 o 1
y_pred = pipeline.predict(X_test)

# ── Accuracy ──────────────────────────────────────────────────────────────────
# Accuracy = (predicciones correctas) / (total de predicciones)
# Como el dataset está balanceado, es una métrica fiable
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy en test: {acc:.4f}  ({acc*100:.2f}%)")
print()

# ── Classification Report ─────────────────────────────────────────────────────
# Métricas más detalladas por clase:
# - Precision:  de los que predije como sarcásticos, ¿cuántos lo eran realmente?
# - Recall:     de todos los sarcásticos reales, ¿cuántos detecté?
# - F1-score:   media armónica de precision y recall (balance entre ambos)
print(classification_report(y_test, y_pred,
                             target_names=['No sarcástico', 'Sarcástico']))

In [ ]:
# ── Matriz de Confusión ───────────────────────────────────────────────────────
# La matriz de confusión muestra cuántas predicciones fueron:
# - Verdaderos Positivos  (TP): predijo Sarcástico, era Sarcástico  ✅
# - Verdaderos Negativos  (TN): predijo No Sarcástico, era No Sarcástico ✅
# - Falsos Positivos (FP): predijo Sarcástico, NO era Sarcástico ❌ (error tipo I)
# - Falsos Negativos (FN): predijo No Sarcástico, SÍ era Sarcástico ❌ (error tipo II)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No sarc.', 'Pred: Sarcástico'],
            yticklabels=['Real: No sarc.', 'Real: Sarcástico'],
            linewidths=0.5)

ax.set_title('Matriz de Confusión', fontsize=14)
ax.set_ylabel('Etiqueta Real')
ax.set_xlabel('Etiqueta Predicha')
plt.tight_layout()
plt.show()

# Interpretación:
# Los cuadros de la diagonal principal (arriba-izquierda y abajo-derecha)
# representan las predicciones CORRECTAS. Cuanto más grandes, mejor.
# Los cuadros fuera de la diagonal son ERRORES del modelo.

## 9. Análisis de los coeficientes: ¿qué palabras predicen el sarcasmo?

En la Regresión Logística, cada palabra tiene un **coeficiente** (peso):
- **Coeficiente positivo alto** → la palabra indica SARCASMO
- **Coeficiente negativo alto** → la palabra indica NO SARCASMO

Visualizar esto nos da interpretabilidad: entendemos *por qué* el modelo toma sus decisiones.

In [ ]:
# Extraemos el vectorizador y el clasificador del pipeline
tfidf_vec = pipeline.named_steps['tfidf']   # el TF-IDF ya entrenado
lr_model  = pipeline.named_steps['clf']     # la Regresión Logística ya entrenada

# Obtenemos el vocabulario (lista de todas las palabras/bigramas)
feature_names = np.array(tfidf_vec.get_feature_names_out())

# Obtenemos los coeficientes del modelo (uno por palabra)
# coef_[0] porque es un problema binario (solo hay una clase de referencia)
coeficientes = lr_model.coef_[0]

print(f"Número de features (palabras/bigramas): {len(feature_names):,}")
print(f"Forma de los coeficientes: {coeficientes.shape}")

In [ ]:
# ── Top 20 palabras más predictivas de SARCASMO ──────────────────────────────
# Ordenamos por coeficiente de mayor a menor y cogemos los 20 primeros
top_n = 20

# Índices ordenados: del coeficiente más alto (sarcástico) al más bajo
top_sarcasmo_idx = np.argsort(coeficientes)[::-1][:top_n]
top_no_sarc_idx  = np.argsort(coeficientes)[:top_n]

# Creamos un DataFrame para visualizar mejor
df_coef_sarc = pd.DataFrame({
    'palabra': feature_names[top_sarcasmo_idx],
    'coeficiente': coeficientes[top_sarcasmo_idx]
})

df_coef_no_sarc = pd.DataFrame({
    'palabra': feature_names[top_no_sarc_idx],
    'coeficiente': coeficientes[top_no_sarc_idx]
})

print("Top 20 palabras/bigramas que predicen SARCASMO:")
print(df_coef_sarc.to_string(index=False))
print()
print("Top 20 palabras/bigramas que predicen NO SARCASMO:")
print(df_coef_no_sarc.to_string(index=False))

In [ ]:
# ── Gráfico de barras horizontales ───────────────────────────────────────────
# Visualizamos los top 15 de cada clase lado a lado

n_show = 15

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel izquierdo: palabras más predictivas de SARCASMO
ax_sarc = axes[0]
palabras_s = feature_names[top_sarcasmo_idx[:n_show]][::-1]
coefs_s    = coeficientes[top_sarcasmo_idx[:n_show]][::-1]
ax_sarc.barh(palabras_s, coefs_s, color='#e74c3c', edgecolor='black')
ax_sarc.set_title('🔴 Top palabras → SARCASMO', fontsize=13)
ax_sarc.set_xlabel('Coeficiente (más alto = más sarcástico)')
ax_sarc.axvline(0, color='black', linewidth=0.8)

# Panel derecho: palabras más predictivas de NO SARCASMO
ax_no = axes[1]
palabras_n = feature_names[top_no_sarc_idx[:n_show]][::-1]
coefs_n    = coeficientes[top_no_sarc_idx[:n_show]][::-1]
ax_no.barh(palabras_n, coefs_n, color='#3498db', edgecolor='black')
ax_no.set_title('🔵 Top palabras → NO SARCASMO', fontsize=13)
ax_no.set_xlabel('Coeficiente (más negativo = más no-sarcástico)')
ax_no.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Palabras más predictivas según la Regresión Logística',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Interpretación:
# Las palabras en rojo son las que el modelo asocia con sarcasmo.
# Las palabras en azul son las que el modelo asocia con comentarios sinceros.
# Los bigramas como "totally not" o "yeah right" capturan patrones sarcásticos.

## 10. Predicciones en comentarios nuevos

Probamos el modelo con frases inventadas para ver si funciona de forma intuitiva.

In [ ]:
# Comentarios de prueba inventados
# Mezcla de comentarios claramente sarcásticos y sinceros
comentarios_prueba = [
    "Oh yeah, because that always works out SO well.",        # sarcástico
    "What a great idea, totally not going to fail.",           # sarcástico
    "Thanks for the help, I really appreciate it.",            # sincero
    "This is the best day ever, nothing could go wrong!",      # sarcástico
    "I disagree with your point, here is my reasoning.",       # sincero
    "Sure, because politicians always tell the truth.",        # sarcástico
    "The weather today is really nice.",                       # sincero
    "Oh brilliant, another Monday. Just what I needed.",       # sarcástico
]

# pipeline.predict() vectoriza y clasifica en un solo paso
predicciones = pipeline.predict(comentarios_prueba)

# pipeline.predict_proba() nos da la PROBABILIDAD de cada clase
# Columna 0 = prob de no sarcástico, Columna 1 = prob de sarcástico
probabilidades = pipeline.predict_proba(comentarios_prueba)

print(f"{'Comentario':<55} {'Predicción':<18} {'Prob. sarcasmo':<15}")
print("-" * 90)

for comentario, pred, proba in zip(comentarios_prueba, predicciones, probabilidades):
    etiqueta = "🔴 Sarcástico" if pred == 1 else "🔵 No sarcástico"
    prob_sarc = proba[1]  # probabilidad de clase 1 (sarcástico)
    print(f"{comentario:<55} {etiqueta:<18} {prob_sarc:.2%}")

## 11. Resumen y conclusiones

### Lo que hemos hecho:

| Paso | Acción | Herramienta |
|------|--------|-------------|
| 1 | Carga y exploración del dataset | `pandas` |
| 2 | Eliminación de valores nulos | `dropna()` |
| 3 | Verificación del balance | `value_counts()` |
| 4 | Análisis exploratorio (EDA) | `matplotlib`, `seaborn` |
| 5 | División train/test | `train_test_split()` |
| 6 | Vectorización TF-IDF | `TfidfVectorizer` |
| 7 | Clasificador | `LogisticRegression` |
| 8 | Evaluación | `accuracy_score`, `classification_report` |
| 9 | Interpretabilidad | Coeficientes del modelo |

### Puntos clave a recordar:
- El **Pipeline** evita *data leakage*: el TF-IDF aprende el vocabulario solo con train
- Los **bigramas** (ngram_range=(1,2)) son importantes para capturar negaciones como "not good"
- La Regresión Logística es **interpretable**: podemos ver qué palabras impulsan cada predicción
- El sarcasmo es un problema difícil: ~75-80% de accuracy es un resultado razonable para texto solo

In [ ]:
# Resumen de métricas final
print("=" * 50)
print("      RESUMEN DEL MODELO")
print("=" * 50)
print(f"  Modelo:       TF-IDF + Regresión Logística")
print(f"  Features:     {len(feature_names):,} palabras/bigramas")
print(f"  Train size:   {len(X_train):,} comentarios")
print(f"  Test size:    {len(X_test):,} comentarios")
print(f"  Accuracy:     {accuracy_score(y_test, y_pred):.4f}")
print("=" * 50)